# Clustering MLB pitchers: reproducible data builder
**Paul Hletko — HDDA Project**

Restart the kernel and **Run All**. No pre-existing CSVs, other notebooks, or custom HTTP downloads are required. Install dependencies once with `%pip install pybaseball pandas numpy` if needed. All source retrieval uses public `pybaseball` functions. Outputs go in `pitcher_data/`; change `OUTDIR` below if desired. Pybaseball's own cache is optional, not an input requirement.

The baseline has 12 arsenal features and 5 outcome features, on common complete-case player IDs. Extended family-specific measurements are descriptive/sensitivity features. Data acquisition may take several minutes. Required-source failures stop the build; optional movement/WAR failures are logged and never replaced by invented measurements. Inspect the final validation summary before using outputs.

**Validation:** this notebook was checked with synthetic API responses, supplied audit records, and an empty output directory. A full live API run was not possible in the authoring environment. No numerical outputs are represented as results of a live rebuild.

In [1]:
BUILD_TAG = "regular-season-coverage95-2026-11-22"
print("Notebook build:", BUILD_TAG)
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import json
import time
import numpy as np
import pandas as pd
try:
    import pybaseball as pb
    from pybaseball.statcast_pitcher import statcast_pitcher_pitch_movement
except ImportError as e:
    raise ImportError("Install dependencies first: %pip install pybaseball pandas numpy") from e

START, END = 2017, 2026
MIN_SEASON_PA = 25
MIN_TOTAL_PA = 100
MIN_CONTACT_BBE = 25
USAGE_CLOSURE_TOL_PP = 1.0  # raw percentages must sum to 100 +/- 1 percentage point
MIN_MEASUREMENT_COVERAGE = 0.95
INCLUDE_EXTENDED = True
FETCH_WAR = True
RETRIES = 2
PAUSE_SECONDS = 1.0
OUTDIR = Path("pitcher_data")
RAW = OUTDIR / "raw"
AUDIT = OUTDIR / "audit"
for directory in [OUTDIR, RAW, AUDIT]:
    directory.mkdir(parents=True, exist_ok=True)
assert START <= END and MIN_SEASON_PA > 0 and MIN_TOTAL_PA >= MIN_SEASON_PA
# Explicit exclusion: conflicting rows without split weights in supplied 2018/2020 data.
EXCLUSIONS = {519381: "Pat Venditte: ambiguous arsenal split rows without weights"}
MAIN = ["FF", "SI", "FC", "SL", "ST", "CU", "CH", "FS"]
FASTBALLS, BREAKING = ["FF", "SI", "FC"], ["SL", "ST", "CU"]
USAGE_COLS = [f"usage_{p}" for p in MAIN + ["other"]]
ARSENAL_CORE = USAGE_COLS + ["arsenal_entropy", "velo_avg", "spin_avg"]
RESULT_CORE = ["res_k_percent", "res_bb_percent", "res_est_woba", "res_avg_hit_speed", "res_brl_percent"]
EXTENDED = ["fb_velo", "fb_spin", "velo_gap", "fb_ivb", "fb_abs_hb", "brk_ivb", "brk_abs_hb"]
ARSENAL_METRICS = ARSENAL_CORE + EXTENDED + ["fb_share", "mean_seasonal_arsenal_breadth"]
CONTACT_METRICS = ["avg_hit_speed", "brl_percent", "ev95percent", "anglesweetspotpercent"]
EXPECTED_METRICS = ["est_woba", "est_ba", "est_slg", "xera"]
collection_log = []
# Regular-season boundaries; 2018 includes Game 163 and 2024 includes the Sep 30 doubleheader.
REGULAR_SEASON_DATES = {
    2017: ("2017-04-02", "2017-10-01"), 2018: ("2018-03-29", "2018-10-01"),
    2019: ("2019-03-20", "2019-09-29"), 2020: ("2020-07-23", "2020-09-27"),
    2021: ("2021-04-01", "2021-10-03"), 2022: ("2022-04-07", "2022-10-05"),
    2023: ("2023-03-30", "2023-10-01"), 2024: ("2024-03-20", "2024-09-30"),
    2025: ("2025-03-18", "2025-09-28"), 2026: ("2026-03-25", "2026-09-27"),
}
MIN_COMMON_PA_COVERAGE = 0.95
COVERAGE_SENSITIVITY = [0.90, 0.95, 1.00]
# Small cross-source discrepancies remain visible, not forced to equality.
# Above this review tolerance, a season cannot enter the shared analysis window.
BF_REVIEW_ABS = 2
BF_REVIEW_REL = 0.005
assert all(year in REGULAR_SEASON_DATES for year in range(START, END + 1))
assert 0 < MIN_COMMON_PA_COVERAGE <= 1
save_dates = pd.DataFrame([{"Season": year, "start": REGULAR_SEASON_DATES[year][0],
                           "end": REGULAR_SEASON_DATES[year][1]}
                          for year in range(START, END + 1)])
save_dates.to_csv(AUDIT / "regular_season_date_windows.csv", index=False)


Notebook build: regular-season-coverage95-2026-11-22


## Regular-season and coverage correction

**Primary rule:** retain a pitcher if baseline measurements and compatible outcome counts jointly cover >=95% of PA from eligible seasons, with >=100 observed PA. Every baseline feature uses precisely the same retained seasons for that pitcher. A small incomplete season no longer invalidates a long record. `PA_total` describes all eligible seasons; `analysis_PA_total`, `analysis_seasons` and `coverage_PA_fraction` describe actual support. Coverage is not silently renormalized in the audit, and no missing season is filled with a league median.

**Source scope:** use `pybaseball.pitching_stats_range` with explicit regular-season start/end dates, instead of the March–November wrapper that can include postseason games. Raw regular-season responses and the date windows are saved. Local date configuration includes 2018 Game 163 and 2024 September 30 make-up games. Residual BF/PA discrepancies are audited: the review tolerance is max(2 BF, 0.5% of season Savant PA), not an alteration of the published count denominator. Larger mismatches exclude that season from usable coverage. This tolerance should be examined in the report.

**Controls:** `pitcher_features_strict_complete.csv` retains 100%-coverage players. `audit/coverage_threshold_sensitivity.csv` reports 90/95/100% sample sizes. `audit/tenure_selection_summary.csv` reports retention by eligible-season count. The extended table recomputes both core and extended features using its own shared usable seasons; compare representations within that table, not across mismatched windows.

Movement collection now tries canonical SI and CU before their legacy aliases, preserving response/schema failures. Source availability is still validated live. These optional fields do not control baseline membership.

Regular-season date references: [2017](https://www.mlb.com/news/mlb-announces-game-times-for-2017-season-c213895888), [2018 Game 163](https://www.mlb.com/news/cubs-lose-nl-central-tiebreaker-to-brewers-c296785202), [2019](https://www.mlb.com/news/what-s-at-stake-in-final-weekend-2019-mlb-season), [2020](https://www.mlb.com/news/marlins-defeat-yankees-to-clinch-winning-season), [2021](https://www.mlb.com/news/what-you-need-to-know-about-mlb-in-2021), [2022](https://www.mlb.com/news/mlb-revised-2022-regular-season-schedule), [2023](https://www.mlb.com/press-release/press-release-mlb-announces-game-times-for-2023-regular-season-schedule), [2024](https://www.baseball-reference.com/leagues/majors/2024-schedule.shtml), [2025](https://www.mlb.com/press-release/press-release-mlb-announces-2025-regular-season-schedule), [2026](https://www.mlb.com/news/mlb-2026-schedule-released).


## Helpers and source validation
Every response is written to `raw/` before transformations. No output CSV is read back as an input. Source schemas are checked explicitly. Unexpected duplicate player records are not averaged or arbitrarily removed. Baseball innings are converted to integer outs **before** summation.

In [2]:
def save(df, filename, directory=OUTDIR):
    df.to_csv(directory / filename, index=False)


def flush_log():
    save(pd.DataFrame(collection_log, columns=["source", "season", "status", "rows", "detail"]), "collection_log.csv", AUDIT)


def fetch(label, year, function, *args, required=True, **kwargs):
    error = None
    for attempt in range(RETRIES):
        try:
            df = function(*args, **kwargs)
            if not isinstance(df, pd.DataFrame) or df.empty:
                raise ValueError("Empty or non-tabular response")
            df = df.copy()
            save(df, f"{label}_{year}.csv", RAW)
            collection_log.append(dict(source=label, season=year, status="ok", rows=len(df), detail=""))
            flush_log()
            return df
        except Exception as e:
            error = e
            if attempt + 1 < RETRIES:
                time.sleep(PAUSE_SECONDS)
    collection_log.append(dict(source=label, season=year, status="failed", rows=0, detail=repr(error)))
    flush_log()
    if required:
        raise RuntimeError(f"Required source {label}, season {year}, failed: {error}. See audit/collection_log.csv") from error
    return None


def col(df, candidates, description):
    found = next((c for c in candidates if c in df.columns), None)
    if found is None:
        raise ValueError(f"Missing {description}: expected {candidates}; received {list(df.columns)}")
    return found


def numeric(values):
    return pd.to_numeric(values, errors="raise")


def unique(df, keys, label):
    if df[keys].isna().any().any() or df.duplicated(keys).any():
        save(df.loc[df[keys].isna().any(axis=1) | df.duplicated(keys, keep=False)], f"duplicates_{label}.csv", AUDIT)
        raise ValueError(f"Missing/duplicate identifiers in {label}; inspect duplicate diagnostic")


def innings_to_outs(values):
    v = numeric(values)
    whole = np.floor(v)
    rem = (v - whole) * 10
    if v.isna().any() or (v < 0).any() or not np.isclose(rem, np.rint(rem)).all() or not np.rint(rem).isin([0, 1, 2]).all():
        raise ValueError("Invalid source baseball innings; expected .0, .1 or .2")
    return (3 * whole + np.rint(rem)).astype("int64")


def wmean(g, field, weight, complete=False):
    valid = g[field].notna() & g[weight].gt(0)
    if not valid.any() or (complete and not valid.all()):
        return np.nan
    return np.average(g.loc[valid, field], weights=g.loc[valid, weight])


def weighted_pitch_summary(values, weights):
    values = values.reindex(index=weights.index, columns=weights.columns)
    observed = values.notna() & weights.gt(0)
    den = weights.where(observed).sum(axis=1)
    coverage = den / weights.sum(axis=1).replace(0, np.nan)
    mean = (values.where(observed) * weights.where(observed)).sum(axis=1, min_count=1) / den.replace(0, np.nan)
    return mean.where(coverage >= MIN_MEASUREMENT_COVERAGE)


assert innings_to_outs(pd.Series([0.1, 1.2, 2.0])).tolist() == [1, 5, 6]

## Pitcher identities, workload and outcome counts
`pitching_stats_bref` supplies SO, BB and BF as well as role statistics. K% and BB% are calculated from counts, not averages of percentages. Source BF is checked against the corresponding Savant season's PA. All discrepancies are logged; differences exceeding the documented review tolerance exclude that season from the shared observed window.
Missing saves are retained and logged because saves are optional descriptors. A multiseason saves total remains missing if any included registry season has missing saves. Negative, infinite or fractional observed saves still stop validation with a row-level diagnostic.


In [3]:
def role_season(year):
    start_date, end_date = REGULAR_SEASON_DATES[year]
    raw = fetch("bref_regular_pitching", year, pb.pitching_stats_range, start_date, end_date)
    mapping = {
        "mlbam_id": ["mlbID", "mlbam_id"], "name": ["Name", "name"],
        "G": ["G"], "GS": ["GS"], "IP": ["IP"], "SV": ["SV"],
        "SO": ["SO", "K"], "BB": ["BB"], "BF": ["BF"],
    }
    df = pd.DataFrame({k: raw[col(raw, v, k)] for k, v in mapping.items()})
    for c in ["mlbam_id", "G", "GS", "IP", "SV", "SO", "BB", "BF"]:
        df[c] = numeric(df[c])
    unique(df, ["mlbam_id"], f"bref_{year}")
    for c in ["mlbam_id", "G", "GS", "SV", "SO", "BB", "BF"]:
        missing = df[c].isna()
        invalid_observed = ~missing & (~np.isfinite(df[c]) | df[c].lt(0) |
                                      ~np.isclose(df[c], np.rint(df[c])))
        bad = invalid_observed | (missing if c != "SV" else False)
        if bad.any():
            diagnostic = f"invalid_role_{c}_{year}.csv"
            save(df.loc[bad], diagnostic, AUDIT)
            raise ValueError(f"{year}: invalid count {c} in {int(bad.sum())} rows; "
                             f"inspect audit/{diagnostic}")
    # Saves are optional descriptors. Do not assume a missing value is zero.
    if df.SV.isna().any():
        save(df.loc[df.SV.isna()], f"missing_saves_{year}.csv", AUDIT)
        print(f"{year}: preserving {int(df.SV.isna().sum())} missing saves values")
    if (df.G <= 0).any() or (df.GS > df.G).any() or (df.SV > df.G - df.GS).any() or (df.SO + df.BB > df.BF).any():
        raise ValueError(f"{year}: inconsistent role or outcome counts")
    df["outs"] = innings_to_outs(df.IP)
    df["Season"] = year
    return df


role_seasons = pd.concat([role_season(year) for year in range(START, END + 1)], ignore_index=True)
save(role_seasons, "pitcher_role_seasons.csv")

2017: preserving 593 missing saves values
2018: preserving 634 missing saves values
2019: preserving 632 missing saves values
2020: preserving 604 missing saves values
2021: preserving 711 missing saves values
2022: preserving 649 missing saves values
2023: preserving 649 missing saves values
2024: preserving 632 missing saves values
2025: preserving 658 missing saves values
2026: preserving 611 missing saves values


## Resolve register metadata
This step is separate so it can be retried after role collection. Only in-sample positive MLBAM IDs are used. Compatible duplicate metadata are combined; conflicting fields remain missing and are logged. Pitcher names remain from the season table.


In [4]:
def resolve_register(raw, wanted_ids):
    """Resolve metadata only: never choose arbitrarily between conflicting identifiers."""
    fields = ["key_mlbam", "key_bbref", "key_fangraphs", "key_retro",
              "mlb_played_first", "mlb_played_last"]
    df = raw.reindex(columns=fields).copy()
    ids = pd.to_numeric(df.key_mlbam, errors="coerce")
    valid = ids.notna() & np.isfinite(ids) & ids.gt(0) & np.isclose(ids, np.rint(ids))
    df = df.loc[valid & ids.isin(wanted_ids)].copy()
    df["key_mlbam"] = ids.loc[df.index].astype("int64")
    for field in ["key_fangraphs", "mlb_played_first", "mlb_played_last"]:
        df[field] = pd.to_numeric(df[field], errors="coerce").mask(lambda values: values <= 0)
    for field in ["key_bbref", "key_retro"]:
        df[field] = df[field].astype("string").str.strip()
        df[field] = df[field].mask(df[field].isin(["", "-1", "0"]))
    duplicates = df.loc[df.duplicated("key_mlbam", keep=False)]
    save(duplicates, "duplicates_chadwick_in_sample.csv", AUDIT)
    records, conflicts = [], []
    for pid, group in df.groupby("key_mlbam"):
        record = {"key_mlbam": int(pid)}
        for field in fields[1:]:
            values = group[field].dropna().unique()
            record[field] = values[0] if len(values) == 1 else np.nan
            if len(values) > 1:
                conflicts.append({"mlbam_id": int(pid), "field": field,
                                  "conflicting_values": json.dumps([str(v) for v in values])})
        records.append(record)
    resolved = pd.DataFrame(records, columns=fields)
    resolved["key_mlbam"] = pd.to_numeric(resolved.key_mlbam).astype("int64")
    for field in ["key_fangraphs", "mlb_played_first", "mlb_played_last"]:
        resolved[field] = pd.to_numeric(resolved[field], errors="coerce")
    unique(resolved, ["key_mlbam"], "resolved_chadwick")
    save(pd.DataFrame(conflicts, columns=["mlbam_id", "field", "conflicting_values"]),
         "chadwick_metadata_conflicts.csv", AUDIT)
    save(pd.DataFrame([{"raw_rows": len(raw), "invalid_or_missing_id_rows": int((~valid).sum()),
                       "in_sample_rows": len(df), "duplicate_in_sample_rows": len(duplicates),
                       "resolved_players": len(resolved), "conflicting_fields": len(conflicts),
                       "players_without_register_match": len(set(wanted_ids) - set(resolved.key_mlbam))}]),
         "chadwick_resolution_summary.csv", AUDIT)
    print(f"Register: {len(resolved)} matched pitchers; {len(duplicates)} duplicate source rows; "
          f"{len(conflicts)} conflicting metadata fields left missing")
    return resolved


registry = fetch("chadwick_register", "all", pb.chadwick_register)
id_col = col(registry, ["key_mlbam"], "Chadwick MLBAM ID")
registry = resolve_register(registry, set(role_seasons.mlbam_id))
rename = {"key_mlbam": "mlbam_id", "key_bbref": "bbref_id", "key_fangraphs": "fangraphs_id",
          "key_retro": "retrosheet_id", "mlb_played_first": "register_first_mlb_season",
          "mlb_played_last": "register_last_mlb_season"}
registry = registry.rename(columns=rename)
reg_cols = [c for c in ["mlbam_id", "bbref_id", "fangraphs_id", "retrosheet_id", "register_first_mlb_season", "register_last_mlb_season"] if c in registry]
pitchers = role_seasons.groupby("mlbam_id", as_index=False).agg(
    name=("name", "first"), window_first_season=("Season", "min"), window_last_season=("Season", "max"),
    seasons_pitched=("Season", "nunique"), G_total=("G", "sum"), GS_total=("GS", "sum"),
    outs_total=("outs", "sum"), SV_total=("SV", lambda values: values.sum(min_count=len(values))))
pitchers = pitchers.merge(registry[reg_cols], on="mlbam_id", how="left", validate="one_to_one")
pitchers["IP_total"] = pitchers.outs_total / 3
pitchers["GS_share"] = pitchers.GS_total / pitchers.G_total
pitchers["IP_per_G"] = pitchers.IP_total / pitchers.G_total
pitchers["SV_per_G"] = pitchers.SV_total / pitchers.G_total
if "fangraphs_id" in pitchers:
    pitchers["fangraphs_id"] = numeric(pitchers.fangraphs_id).mask(lambda s: s <= 0).astype("Int64")
# Preserve register dates as source metadata; never infer retirement from last observation.
if "register_last_mlb_season" in pitchers:
    pitchers["last_mlb_season_observed"] = pitchers[["register_last_mlb_season", "window_last_season"]].max(axis=1)
    save(pitchers.loc[pitchers.register_last_mlb_season < pitchers.window_last_season], "register_date_conflicts.csv", AUDIT)
print(f"Registry: {len(pitchers)} unique pitchers")

Register: 2474 matched pitchers; 0 duplicate source rows; 0 conflicting metadata fields left missing
Registry: 2474 unique pitchers


## Optional career WAR
WAR is for interpretation only. Exact duplicate source rows are removed; conflicting duplicate stint keys are rejected. If the source supplies both a season-total row and team stints, only the total row is used for that player-season. Missing WAR is kept missing. A failed WAR request does not block core feature collection.

In [5]:
def war_totals():
    raw = fetch("bwar_pitch", "all", pb.bwar_pitch, return_all=True, required=False)
    if raw is None:
        return pd.DataFrame(columns=["mlbam_id", "career_WAR", "window_WAR"])
    try:
        df = raw.drop_duplicates().rename(columns={"mlb_ID": "mlbam_id", "year_ID": "Season"}).copy()
        for c in ["mlbam_id", "Season", "WAR"]:
            df[c] = numeric(df[c])
        df = df.dropna(subset=["mlbam_id", "Season"])
        keys = [c for c in ["mlbam_id", "Season", "team_ID", "stint_ID"] if c in df]
        unique(df, keys, "war_stints")
        parts = []
        for _, g in df.groupby(["mlbam_id", "Season"]):
            if "team_ID" in g:
                total = g.team_ID.astype(str).str.upper().str.fullmatch(r"TOT|[2-9]TM")
                if total.any():
                    if total.sum() != 1:
                        raise ValueError("Multiple WAR season totals")
                    g = g.loc[total]
            parts.append({"mlbam_id": g.mlbam_id.iloc[0], "Season": g.Season.iloc[0],
                          "WAR": g.WAR.sum(min_count=len(g))})
        by_season = pd.DataFrame(parts)
        save(by_season, "pitcher_war_seasons.csv")
        records = []
        for pid, g in by_season.groupby("mlbam_id"):
            win = g.loc[g.Season.between(START, END), "WAR"]
            records.append({"mlbam_id": pid, "career_WAR": g.WAR.sum(min_count=len(g)),
                            "window_WAR": win.sum(min_count=len(win)) if len(win) else np.nan})
        return pd.DataFrame(records)
    except Exception as e:
        collection_log.append(dict(source="war_validation", season="all", status="failed", rows=0, detail=repr(e)))
        flush_log()
        print("WAR unavailable after validation:", e)
        return pd.DataFrame(columns=["mlbam_id", "career_WAR", "window_WAR"])


war = war_totals() if FETCH_WAR else pd.DataFrame(columns=["mlbam_id", "career_WAR", "window_WAR"])
pitchers = pitchers.merge(war, on="mlbam_id", how="left", validate="one_to_one").sort_values("name").reset_index(drop=True)
save(pitchers, "pitchers.csv")
save(pd.DataFrame([{"mlbam_id": k, "reason": v} for k, v in EXCLUSIONS.items()]), "statcast_exclusions.csv", AUDIT)

## Raw arsenal coverage and extended measurements
Pitch-type queries use `minPA=0`; pitcher workload eligibility is enforced locally. Both minPA=0 and minPA=25 are audited for **every year**. Only raw usage totals within 99–101% are normalized; incomplete arsenals remain missing. This is a coverage diagnostic, not proof that every pitch was classified correctly.

A weighted physical summary needs measurements covering at least 95% of relevant usage. Zero usage of a family leaves its physical summary undefined. Extended movement is optional and must explicitly identify induced vertical break; total vertical break is never relabeled as induced break.

In [6]:
def usage_table(raw, year, label):
    df = raw.copy()
    df["player_id"] = numeric(df[col(df, ["player_id", "pitcher", "pitcher_id"], "arsenal ID")])
    df = df.loc[~df.player_id.isin(EXCLUSIONS)].copy()
    df["pitch_type"] = df["pitch_type"].astype("string").str.upper()
    unique(df, ["player_id", "pitch_type"], f"usage_{label}_{year}")
    df["pitch_usage"] = numeric(df["pitch_usage"])
    wide = df.pivot(index="player_id", columns="pitch_type", values="pitch_usage")
    sums = wide.sum(axis=1, min_count=1)
    audit = pd.DataFrame({"player_id": sums.index, "raw_usage_percent_sum": sums.values,
                          "reported_pitch_types": wide.notna().sum(axis=1).values}).set_index("player_id")
    bad_values = wide.lt(0).any(axis=1) | wide.gt(100).any(axis=1)
    missing_supplied = df.assign(missing=df.pitch_usage.isna()).groupby("player_id").missing.any()
    audit["valid_closure"] = sums.sub(100).abs().le(USAGE_CLOSURE_TOL_PP) & ~bad_values & ~missing_supplied
    save(audit.reset_index(), f"usage_coverage_{label}_{year}.csv", AUDIT)
    use = wide.fillna(0).div(sums, axis=0)
    use.loc[~audit.valid_closure] = np.nan
    return use, audit


def wide_measure(raw, suffix, label):
    df = raw.copy()
    idc = col(df, ["pitcher", "player_id", "pitcher_id"], "measurement pitcher ID")
    df[idc] = numeric(df[idc])
    df = df.loc[~df[idc].isin(EXCLUSIONS)]
    unique(df, [idc], label)
    fields = [c for c in df if c.endswith(suffix)]
    if not fields:
        raise ValueError(f"No fields ending {suffix} in {label}: {list(df.columns)}")
    out = df.set_index(idc)[fields].apply(numeric)
    out.columns = [c[:-len(suffix)].upper() for c in fields]
    return out


def movement_tables(year):
    frames = []
    for pt in MAIN:
        codes = [pt] + ({"SI": ["SIFT"], "CU": ["CUKC"]}.get(pt, []))
        for code in codes:
            raw = fetch(f"movement_{pt}_{code}", year, statcast_pitcher_pitch_movement, year,
                        minP=0, pitch_type=code, required=False)
            if raw is None:
                continue
            try:
                idc = col(raw, ["pitcher_id", "pitcher", "player_id"], "movement ID")
                ivbc = col(raw, ["pitcher_break_z_induced", "break_z_induced"], "explicit induced vertical break")
                hbc = col(raw, ["pitcher_break_x", "break_x"], "horizontal break")
                d = raw[[idc, ivbc, hbc]].copy()
                d.columns = ["player_id", "ivb", "hb"]
                d = d.apply(numeric)
                d = d.loc[~d.player_id.isin(EXCLUSIONS)]
                unique(d, ["player_id"], f"movement_{pt}_{year}")
                d["pitch_type"] = pt
                d["hb"] = d.hb.abs()
                frames.append(d)
                break
            except Exception as e:
                collection_log.append(dict(source=f"movement_schema_{pt}_{code}", season=year,
                                           status="failed", rows=0, detail=repr(e)))
                flush_log()
    if not frames:
        return pd.DataFrame(), pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    return (df.pivot(index="player_id", columns="pitch_type", values="ivb"),
            df.pivot(index="player_id", columns="pitch_type", values="hb"))


def arsenal_season(year):
    raw = fetch("arsenal_min0", year, pb.statcast_pitcher_arsenal_stats, year, minPA=0)
    use, audit = usage_table(raw, year, "min0")
    old = fetch("arsenal_min25", year, pb.statcast_pitcher_arsenal_stats, year, minPA=25, required=False)
    if old is not None:
        _, a25 = usage_table(old, year, "min25")
        comparison = audit.add_suffix("_min0").join(a25.add_suffix("_min25"), how="outer")
        save(comparison.reset_index(), f"usage_threshold_comparison_{year}.csv", AUDIT)
    if not audit.valid_closure.any():
        raise ValueError(f"{year}: no valid usage totals near 100%; check raw fields/units")
    speed = wide_measure(fetch("arsenal_speed", year, pb.statcast_pitcher_pitch_arsenal,
                               year, minP=0, arsenal_type="avg_speed"), "_avg_speed", f"speed_{year}")
    spin = wide_measure(fetch("arsenal_spin", year, pb.statcast_pitcher_pitch_arsenal,
                              year, minP=0, arsenal_type="avg_spin"), "_avg_spin", f"spin_{year}")
    f = pd.DataFrame(index=use.index)
    for pt in MAIN:
        f[f"usage_{pt}"] = use[pt] if pt in use else 0.0
    other = [c for c in use if c not in MAIN]
    f["usage_other"] = use[other].sum(axis=1) if other else 0.0
    f[USAGE_COLS] = f[USAGE_COLS].mask(f[USAGE_COLS].abs() < 1e-10, 0.0)
    f["arsenal_entropy"] = -(use.where(use > 0) * np.log(use.where(use > 0))).sum(axis=1)
    f["mean_seasonal_arsenal_breadth"] = use.ge(.05).sum(axis=1)
    f["velo_avg"] = weighted_pitch_summary(speed, use)
    f["spin_avg"] = weighted_pitch_summary(spin, use)
    fb = use.reindex(columns=FASTBALLS).fillna(0)
    br = use.reindex(columns=BREAKING).fillna(0)
    f["fb_share"] = fb.sum(axis=1)
    for c in EXTENDED:
        f[c] = np.nan
    if INCLUDE_EXTENDED:
        ivb, hb = movement_tables(year)
        f["fb_velo"] = weighted_pitch_summary(speed, fb)
        f["fb_spin"] = weighted_pitch_summary(spin, fb)
        f["velo_gap"] = f.fb_velo - weighted_pitch_summary(speed, use[[c for c in use if c not in FASTBALLS]])
        f["fb_ivb"] = weighted_pitch_summary(ivb, fb)
        f["fb_abs_hb"] = weighted_pitch_summary(hb, fb)
        f["brk_ivb"] = weighted_pitch_summary(ivb, br)
        f["brk_abs_hb"] = weighted_pitch_summary(hb, br)
    f.loc[~audit.valid_closure] = np.nan
    f["qa_usage_valid"] = audit.valid_closure
    f.index.name = "player_id"
    return f.reset_index()

## Build eligible pitcher-seasons
The minimum 25 PA is applied explicitly even if the server ignores its qualification parameter. The same resulting player-season records define all feature representations. Role/outcome counts are joined by MLBAM ID and season. Missing baseline measurements remain missing; no missing family is assigned an artificial velocity or movement.

In [7]:
def statcast_season(year):
    exp = fetch("expected_stats", year, pb.statcast_pitcher_expected_stats, year, minPA=0)
    exp["player_id"] = numeric(exp["player_id"])
    exp["pa"] = numeric(exp["pa"])
    unique(exp, ["player_id"], f"expected_{year}")
    for c in EXPECTED_METRICS:
        exp[c] = numeric(exp[c])
    eligible = exp.pa.ge(MIN_SEASON_PA) & ~exp.player_id.isin(EXCLUSIONS)
    save(exp.loc[~eligible].assign(Season=year), f"excluded_expected_seasons_{year}.csv", AUDIT)
    exp = exp.loc[eligible, ["player_id", "pa"] + EXPECTED_METRICS].copy()
    ev = fetch("exitvelo_barrels", year, pb.statcast_pitcher_exitvelo_barrels, year, minBBE=0)
    ev["player_id"] = numeric(ev["player_id"])
    unique(ev, ["player_id"], f"contact_{year}")
    for c in CONTACT_METRICS:
        if c not in ev:
            ev[c] = np.nan
        ev[c] = numeric(ev[c])
    bbe_col = col(ev, ["attempts", "batted_ball_events", "bbe", "bip"], "contact BBE count")
    ev["contact_BBE"] = numeric(ev[bbe_col])
    # Explicit contact reliability threshold; preserve original response in raw/.
    ev.loc[ev.contact_BBE < MIN_CONTACT_BBE, CONTACT_METRICS] = np.nan
    ars = arsenal_season(year)
    role = role_seasons.loc[role_seasons.Season.eq(year), ["mlbam_id", "name", "BF", "SO", "BB"]]
    d = exp.merge(ev[["player_id", "contact_BBE"] + CONTACT_METRICS], on="player_id", how="left", validate="one_to_one")
    d = d.merge(ars, on="player_id", how="left", validate="one_to_one")
    d = d.merge(role, left_on="player_id", right_on="mlbam_id", how="left", validate="one_to_one")
    d["Season"] = year
    d["qa_bf_matches_pa"] = d.BF.notna() & np.isclose(d.BF, d.pa)
    d["BF_minus_savant_PA"] = d.BF - d.pa
    tolerance = np.maximum(BF_REVIEW_ABS, BF_REVIEW_REL * d.pa)
    d["qa_bf_compatible"] = d.BF.gt(0) & d.SO.notna() & d.BB.notna() & d.BF_minus_savant_PA.abs().le(tolerance)
    # Calendar alignment is enforced at retrieval. The tolerance flags residual
    # reporting differences; retain original SO/BB/BF rather than altering counts.
    save(d.loc[~d.qa_bf_matches_pa], f"outcome_bf_mismatches_{year}.csv", AUDIT)
    d["qa_xera_review"] = d.xera.notna() & (~d.xera.between(0, 20))
    d["qa_usage_valid"] = d.qa_usage_valid.fillna(False).astype(bool)
    print(f"{year}: {len(d)} eligible pitcher-seasons; valid arsenal={int(d.qa_usage_valid.sum())}; "
          f"matched BF={int(d.qa_bf_matches_pa.sum())}")
    baseline_fields = ARSENAL_CORE + ["est_woba", "avg_hit_speed", "brl_percent"]
    d["qa_common_usable"] = d[baseline_fields].notna().all(axis=1) & d.qa_usage_valid & d.qa_bf_compatible
    return d


season_frames = []
for year in range(START, END + 1):
    try:
        season_frames.append(statcast_season(year))
    except Exception as e:
        save(pd.concat(season_frames, ignore_index=True) if season_frames else pd.DataFrame(),
             "pitcher_statcast_seasons_partial.csv", AUDIT)
        collection_log.append(dict(source="season_build", season=year, status="failed", rows=0, detail=repr(e)))
        flush_log()
        raise RuntimeError(f"Build stopped at {year}: {e}. Final feature files were not generated by this run.") from e
statcast_seasons = pd.concat(season_frames, ignore_index=True)
unique(statcast_seasons, ["player_id", "Season"], "statcast_seasons")
assert statcast_seasons.pa.ge(MIN_SEASON_PA).all()
save(statcast_seasons, "pitcher_statcast_seasons.csv")
save(statcast_seasons.loc[statcast_seasons.player_id.eq(641853) | statcast_seasons.qa_xera_review], "xera_review_seasons.csv", AUDIT)
save(statcast_seasons[["player_id", "Season", "pa", "BF", "SO", "BB", "BF_minus_savant_PA", "qa_bf_matches_pa", "qa_bf_compatible", "qa_common_usable"]], "pitcher_outcome_seasons.csv")

2017: 673 eligible pitcher-seasons; valid arsenal=646; matched BF=639
2018: 696 eligible pitcher-seasons; valid arsenal=662; matched BF=663
2019: 712 eligible pitcher-seasons; valid arsenal=676; matched BF=711
2020: 583 eligible pitcher-seasons; valid arsenal=532; matched BF=583
2021: 737 eligible pitcher-seasons; valid arsenal=688; matched BF=716
2022: 724 eligible pitcher-seasons; valid arsenal=678; matched BF=707
2023: 719 eligible pitcher-seasons; valid arsenal=681; matched BF=706
2024: 717 eligible pitcher-seasons; valid arsenal=684; matched BF=692
2025: 734 eligible pitcher-seasons; valid arsenal=701; matched BF=706
2026: 731 eligible pitcher-seasons; valid arsenal=696; matched BF=713


## Aggregate, screen, and export
Total eligibility (100 PA) is applied **after** season filtering. All baseline features use the same usable seasons, covering at least 95% of eligible PA; no missing values are imputed. Continuous summaries use PA weights; contact and arsenal weighting are documented approximations, not exact pooled pitch/batted-ball averages. K% and BB% pool published regular-season counts on that shared observed window. Small residual BF/PA differences remain recorded, with a per-season review tolerance of max(2 BF, 0.5% of Savant PA).

xERA is descriptive only. A review threshold of 20 is not a biological/statistical upper bound: extreme estimates can occur in tiny samples. Raw values are preserved; reviewed xERA is quarantined in screened output, never used as a baseline feature. `career_WAR` includes all available career seasons, whereas `window_WAR` uses the calendar window, not only the eligible Statcast seasons.

In [8]:
def aggregate_features(seasons, extended_mode=False):
    rows, exclusions = [], []
    for pid, g in seasons.groupby("player_id"):
        eligible_pa = g.pa.sum()
        if eligible_pa < MIN_TOTAL_PA:
            exclusions.append({"mlbam_id": pid, "reason": "eligible PA below total minimum", "PA_total": eligible_pa})
            continue
        usable = g.qa_common_usable.copy()
        if extended_mode:
            usable &= g[EXTENDED].notna().all(axis=1)
        observed = g.loc[usable]
        observed_pa = observed.pa.sum()
        coverage = observed_pa / eligible_pa
        r = {"mlbam_id": pid, "PA_total": eligible_pa, "analysis_PA_total": observed_pa,
             "coverage_PA_fraction": coverage, "savant_seasons": len(g), "analysis_seasons": len(observed),
             "analysis_first_season": observed.Season.min() if len(observed) else np.nan,
             "analysis_last_season": observed.Season.max() if len(observed) else np.nan,
             "qa_arsenal_seasons_available": int(g.qa_usage_valid.sum()),
             "qa_xera_seasons_review": int(observed.qa_xera_review.sum()),
             "qa_primary_coverage_eligible": bool(coverage + 1e-12 >= MIN_COMMON_PA_COVERAGE and observed_pa >= MIN_TOTAL_PA),
             "qa_strict_complete": bool(coverage + 1e-12 >= 1.0 and observed_pa >= MIN_TOTAL_PA)}
        for c in ARSENAL_METRICS:
            r[c] = wmean(observed, c, "pa", complete=True)
        for c in EXPECTED_METRICS + CONTACT_METRICS:
            r[f"res_{c}"] = wmean(observed, c, "pa", complete=True)
        for target, count in [("res_k_percent", "SO"), ("res_bb_percent", "BB")]:
            r[target] = 100 * observed[count].sum() / observed.BF.sum() if len(observed) else np.nan
            r[target + "_seasons_observed"] = len(observed)
        r["outcome_BF_total"] = observed.BF.sum() if len(observed) else np.nan
        r["analysis_BF_PA_difference"] = observed.BF.sum() - observed_pa if len(observed) else np.nan
        rows.append(r)
    if not extended_mode:
        save(pd.DataFrame(exclusions, columns=["mlbam_id", "reason", "PA_total"]), "excluded_total_workload.csv", AUDIT)
    if not rows:
        raise ValueError("No pitchers meet total workload eligibility")
    return pd.DataFrame(rows)


features = pitchers.merge(aggregate_features(statcast_seasons), on="mlbam_id", how="inner", validate="one_to_one")
features = features.sort_values("name").reset_index(drop=True)
save(features, "pitcher_features_before_screening.csv")
features[USAGE_COLS] = features[USAGE_COLS].mask(features[USAGE_COLS].abs() < 1e-10, 0.0)
flags = features[["mlbam_id", "name", "PA_total", "analysis_PA_total", "coverage_PA_fraction", "savant_seasons", "analysis_seasons"]].copy()
totals = features[USAGE_COLS].sum(axis=1, min_count=1)
flags["usage_closed"] = totals.notna() & np.isclose(totals, 1.0, atol=1e-8)
flags["usage_valid_range"] = ~(features[USAGE_COLS].lt(0).any(axis=1) | features[USAGE_COLS].gt(1).any(axis=1))
flags["core_complete"] = features[ARSENAL_CORE + RESULT_CORE].notna().all(axis=1)
flags["core_eligible"] = flags.usage_closed & flags.usage_valid_range & flags.core_complete & features.qa_primary_coverage_eligible
flags["strict_eligible"] = flags.core_eligible & features.qa_strict_complete
flags["xera_review"] = features.qa_xera_seasons_review.gt(0)
features["res_xera_source"] = features.res_xera
features.loc[flags.xera_review, "res_xera"] = np.nan
save(flags, "pitcher_feature_quality_flags.csv", AUDIT)
save(features, "pitcher_features.csv")
core = features.loc[flags.core_eligible].copy()
strict = features.loc[flags.strict_eligible].copy()
# Compatibility filename: complete FEATURES, with explicit >=95% shared PA coverage.
save(core, "pitcher_features_core_complete.csv")
save(strict, "pitcher_features_strict_complete.csv")
extended_all = pitchers.merge(aggregate_features(statcast_seasons, extended_mode=True), on="mlbam_id", how="inner", validate="one_to_one")
extended = extended_all.loc[extended_all.qa_primary_coverage_eligible & extended_all[ARSENAL_CORE + RESULT_CORE + EXTENDED].notna().all(axis=1)].copy()
save(extended, "pitcher_features_extended_complete.csv")
# Extended comparisons must use this table for BOTH its core and extended representations.
threshold_rows = []
for threshold in COVERAGE_SENSITIVITY:
    selected = flags.core_complete & flags.usage_closed & flags.usage_valid_range & features.analysis_PA_total.ge(MIN_TOTAL_PA) & features.coverage_PA_fraction.ge(threshold - 1e-12)
    threshold_rows.append({"minimum_coverage": threshold, "pitchers": int(selected.sum()),
                           "median_eligible_seasons": features.loc[selected, "savant_seasons"].median()})
save(pd.DataFrame(threshold_rows), "coverage_threshold_sensitivity.csv", AUDIT)
# All candidate pitchers are retained in this audit, with explicit coverage and membership.
save(flags, "analysis_sample_membership.csv", AUDIT)
tenure = flags.groupby("savant_seasons", as_index=False).agg(
    eligible_pitchers=("mlbam_id", "size"), primary_retained=("core_eligible", "sum"),
    strict_retained=("strict_eligible", "sum"), median_PA_coverage=("coverage_PA_fraction", "median"))
tenure["primary_retention_fraction"] = tenure.primary_retained / tenure.eligible_pitchers
save(tenure, "tenure_selection_summary.csv", AUDIT)
feature_sets = {"arsenal": ARSENAL_CORE, "results": RESULT_CORE, "combined": ARSENAL_CORE + RESULT_CORE}
configuration = {"baseline": feature_sets, "extended_arsenal": ARSENAL_CORE + EXTENDED,
    "baseline_input": "pitcher_features_core_complete.csv", "strict_input": "pitcher_features_strict_complete.csv",
    "extended_input": "pitcher_features_extended_complete.csv", "minimum_common_PA_coverage": MIN_COMMON_PA_COVERAGE,
    "coverage_sensitivity": COVERAGE_SENSITIVITY,
    "season_selection": "One shared usable-season mask per pitcher for every baseline representation; >=95% of eligible PA and >=100 observed PA",
    "regular_season_dates": {str(y): REGULAR_SEASON_DATES[y] for y in range(START, END + 1)},
    "BF_review_tolerance": {"absolute": BF_REVIEW_ABS, "relative": BF_REVIEW_REL, "rule": "max(absolute, relative * Savant PA) per season"},
    "excluded_from_baseline": ["role/workload/IDs/WAR", "xERA/xBA/xSLG", "hard-hit/sweet-spot rates", "extended family features"],
    "season_eligibility": MIN_SEASON_PA, "total_eligibility": MIN_TOTAL_PA,
    "weighting": "Same observed seasons for all baseline features. Continuous summaries PA-weighted; K/BB from actual source SO/BB/BF. Unobserved seasons are not imputed.",
    "whiff": "Only raw pitch-type whiff data retained; no overall whiff rate constructed."}
(OUTDIR / "pitcher_feature_sets.json").write_text(json.dumps(configuration, indent=2))


2780

## Final validation, feature dictionary and run manifest
A nonempty primary table must pass every listed check; 90%, 95% and 100% coverage counts are exported for sensitivity analysis. Empty extended output is permitted if optional movement could not be obtained. Counts are descriptive audit results, not evidence for cluster structure.

In [9]:
checks = {
    "all_required_seasons_collected": set(statcast_seasons.Season) == set(range(START, END + 1)),
    "unique_final_ids": features.mlbam_id.is_unique,
    "nonempty_baseline": len(core) > 0,
    "no_baseline_missing": not core[ARSENAL_CORE + RESULT_CORE].isna().any().any(),
    "finite_baseline": np.isfinite(core[ARSENAL_CORE + RESULT_CORE].to_numpy(float)).all(),
    "closed_baseline_usage": np.isclose(core[USAGE_COLS].sum(axis=1), 1, atol=1e-8).all(),
    "valid_expected_ranges": core.res_est_woba.between(0, 2).all(),
    "valid_rate_ranges": core[["res_k_percent", "res_bb_percent", "res_brl_percent"]].ge(0).all().all() and core[["res_k_percent", "res_bb_percent", "res_brl_percent"]].le(100).all().all(),
    "positive_velocity_spin": core[["velo_avg", "spin_avg", "res_avg_hit_speed"]].gt(0).all().all(),
    "valid_innings_arithmetic": np.isclose(pitchers.IP_total, pitchers.outs_total / 3).all(),
    "season_minimum_enforced": statcast_seasons.pa.ge(MIN_SEASON_PA).all(),
    "total_minimum_enforced": core.PA_total.ge(MIN_TOTAL_PA).all(),
    "common_PA_coverage_sufficient": core.coverage_PA_fraction.ge(MIN_COMMON_PA_COVERAGE - 1e-12).all(),
    "observed_PA_minimum_enforced": core.analysis_PA_total.ge(MIN_TOTAL_PA).all(),
    "outcome_denominators_positive": core.outcome_BF_total.gt(0).all(),
    "count_pooled_K_BB": all(np.isclose(core[c] * core.outcome_BF_total / 100,
                                      np.rint(core[c] * core.outcome_BF_total / 100)).all()
                             for c in ["res_k_percent", "res_bb_percent"]),
    "analysis_seasons_do_not_exceed_eligible": core.analysis_seasons.le(core.savant_seasons).all(),
}
validation = pd.DataFrame([{"check": k, "passed": bool(v)} for k, v in checks.items()])
save(validation, "final_validation.csv", AUDIT)
flow = pd.DataFrame({"stage": ["registry", "eligible player-seasons", "total-workload pitchers", "primary coverage sample", "extended coverage sample"],
                     "count": [len(pitchers), len(statcast_seasons), len(features), len(core), len(extended)]})
save(flow, "sample_flow.csv", AUDIT)
missingness = features[ARSENAL_CORE + RESULT_CORE + EXTENDED].isna().mean().rename("missing_fraction").rename_axis("feature").reset_index()
save(missingness, "feature_missingness.csv", AUDIT)
for name, cols in feature_sets.items():
    save(core[cols].corr().rename_axis("feature").reset_index(), f"correlations_{name}.csv", AUDIT)
records = []
for c in USAGE_COLS:
    records.append({"feature": c, "meaning": "Mean seasonal share of pitches of this type; other combines remaining types", "unit": "fraction", "aggregation": "PA-weighted seasonal share", "baseline": True})
definition = {
    "arsenal_entropy": ("Mean seasonal Shannon entropy of full pitch mix", "nats"),
    "velo_avg": ("Usage-weighted pitch velocity within season", "mph"),
    "spin_avg": ("Usage-weighted pitch spin within season", "rpm"),
    "res_k_percent": ("100 * pooled strikeouts / pooled batters faced", "percentage points"),
    "res_bb_percent": ("100 * pooled walks / pooled batters faced", "percentage points"),
    "res_est_woba": ("Expected weighted on-base average allowed", "wOBA scale"),
    "res_avg_hit_speed": ("Average exit velocity allowed", "mph"),
    "res_brl_percent": ("Barrels as a percentage of batted-ball events", "percentage points"),
    "fb_velo": ("Velocity for FF/SI/FC fastball family", "mph"),
    "fb_spin": ("Spin for FF/SI/FC fastball family", "rpm"),
    "velo_gap": ("Fastball velocity minus ALL nonfastball velocity, including breaking balls", "mph"),
    "fb_ivb": ("Fastball induced vertical break", "source movement units; verify raw source"),
    "fb_abs_hb": ("Magnitude of fastball horizontal break; direction discarded", "source movement units; verify raw source"),
    "brk_ivb": ("SL/ST/CU induced vertical break", "source movement units; verify raw source"),
    "brk_abs_hb": ("Magnitude of SL/ST/CU horizontal break; direction discarded", "source movement units; verify raw source"),
    "mean_seasonal_arsenal_breadth": ("Mean number of pitch types with >=5% seasonal usage", "mean count"),
}
for c, (meaning, unit) in definition.items():
    records.append({"feature": c, "meaning": meaning, "unit": unit,
                    "aggregation": "pooled counts" if c in ["res_k_percent", "res_bb_percent"] else "PA-weighted over shared usable seasons; PA coverage recorded",
                    "baseline": c in ARSENAL_CORE + RESULT_CORE})
save(pd.DataFrame(records), "feature_dictionary.csv")
packages = {}
for package in ["pybaseball", "pandas", "numpy"]:
    try:
        packages[package] = version(package)
    except PackageNotFoundError:
        packages[package] = "unknown"
manifest = {"completed_at_utc": datetime.now(timezone.utc).isoformat(), "packages": packages,
    "years": [START, END], "minimum_season_PA": MIN_SEASON_PA, "minimum_total_PA": MIN_TOTAL_PA,
    "minimum_contact_BBE": MIN_CONTACT_BBE, "usage_closure_tolerance_percentage_points": USAGE_CLOSURE_TOL_PP,
    "minimum_measurement_usage_coverage": MIN_MEASUREMENT_COVERAGE,
    "build_tag": BUILD_TAG, "minimum_common_PA_coverage": MIN_COMMON_PA_COVERAGE,
    "validation_passed": bool(all(checks.values())), "feature_configuration": configuration,
    "counts": dict(zip(flow.stage, flow['count']))}
(OUTDIR / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
flush_log()
print(validation.to_string(index=False))
print(flow.to_string(index=False))
print(pd.DataFrame(threshold_rows).to_string(index=False))
if not all(checks.values()):
    raise RuntimeError("Final validation failed. Do not use these outputs until audit/final_validation.csv is resolved.")
print(f"SUCCESS. Use {OUTDIR / 'pitcher_features_core_complete.csv'} with pitcher_feature_sets.json.")

                                  check  passed
         all_required_seasons_collected    True
                       unique_final_ids    True
                      nonempty_baseline    True
                    no_baseline_missing    True
                        finite_baseline    True
                  closed_baseline_usage    True
                  valid_expected_ranges    True
                      valid_rate_ranges    True
                 positive_velocity_spin    True
               valid_innings_arithmetic    True
                season_minimum_enforced    True
                 total_minimum_enforced    True
          common_PA_coverage_sufficient    True
           observed_PA_minimum_enforced    True
          outcome_denominators_positive    True
                      count_pooled_K_BB    True
analysis_seasons_do_not_exceed_eligible    True
                   stage  count
                registry   2474
 eligible player-seasons   7026
 total-workload pitchers   1638
 primary

## Output guide and sources

| Output | Purpose |
|---|---|
| `pitchers.csv` | Identity, role, innings and optional WAR registry |
| `pitcher_role_seasons.csv` | Per-season source counts underlying role and K/BB calculations |
| `pitcher_statcast_seasons.csv` | Eligible pitcher-seasons plus the shared usable-season mask |
| `pitcher_outcome_seasons.csv` | SO/BB/BF and source-denominator audit |
| `pitcher_features_before_screening.csv` | Aggregated features before descriptive xERA quarantine |
| `pitcher_features.csv` | Full screened feature table, including incomplete records |
| `pitcher_features_core_complete.csv` | Primary >=95%-coverage sample with complete feature values |
| `pitcher_features_extended_complete.csv` | Common extended sensitivity sample |
| `pitcher_feature_sets.json` | Explicit feature lists for `project.ipynb` |
| `feature_dictionary.csv` | Feature definitions, units and aggregation |
| `run_manifest.json` | Parameters, versions, sample counts and validation status |
| `raw/` | Exact API responses, including pitch-type whiff measurements if supplied |
| `audit/` | Collection log, exclusions, per-year coverage comparisons, source mismatches, xERA review, sample flow and final checks |

Load the **core-complete** CSV for all three baseline representations. Read `configuration['baseline']` from `pitcher_feature_sets.json`; do not select all numeric or `res_` columns. PCA, scaling, clustering and stability analysis belong in `project.ipynb`.

Data: [Baseball Savant](https://baseballsavant.mlb.com/), [Baseball Reference](https://www.baseball-reference.com/), [Chadwick Register](https://github.com/chadwickbureau/register), accessed through [pybaseball](https://github.com/jldbc/pybaseball). API references: [pitcher leaderboards](https://github.com/jldbc/pybaseball/blob/master/docs/statcast_pitcher.md), [league pitching functions](https://github.com/jldbc/pybaseball/blob/master/pybaseball/league_pitching_stats.py). Record actual retrieval dates from the manifest in the report.